# From Effects to Decisions
## P@S Class 19: The Green school on the limits of A/B

You estimated the treatment effect. Now what do you DO with it?

**Data labels:**
- Sections 1 and 4 use **REAL** data loaded from Harvard Dataverse (Kalla-Broockman 2018 replication + Offer-Westort 2021 Study 1).
- Sections 2 and 3 use **SIMULATED** data, parameterized to the papers' reported characteristics, because the underlying raw data is either not released (Lewis-Rao) or is a formula with no data (Wheeler).

**Modularity:** each section is self-contained. Run the imports cell (Cell 1) first; after that, any section can be run standalone without the others.

This notebook walks through four moves from today's lecture:

1. **Kalla-Broockman (2018)** [REAL]: meta-analysis of 44 general-election field experiments with DerSimonian-Laird random-effects, showing pooled persuasion effect is ~0.
2. **Portable Power (Wheeler 1974)** [FORMULA]: plug four everyday A/B scenarios into $N \approx 16\sigma^2/\Delta^2$, watch required sample size span five orders of magnitude.
3. **Lewis-Rao (2015)** [SIMULATED]: power simulation showing why ROI CIs stay huge even with millions of person-weeks of data.
4. **Offer-Westort, Coppock & Green (2021)** [REAL]: adaptive 8-arm bandit on right-to-work messaging. Watch Thompson sampling concentrate traffic on the winning arm batch by batch.

Readings for today:
- Kalla & Broockman (2018). "Minimal Persuasive Effects of Campaign Contact." APSR 112(1).
- Lewis & Rao (2015). "Unfavorable Economics of Measuring Ad Returns." QJE 130(4).
- Offer-Westort, Coppock & Green (2021). "Adaptive Experimental Design." AJPS 65(4).
- Wheeler, R. E. (1974). "Portable Power." Technometrics 16(2).

In [ ]:
# Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)  # reproducibility
plt.style.use('seaborn-v0_8-whitegrid')

# Consistent color palette
BLUE, RED, GREEN, GRAY, ORANGE = '#2980b9', '#c0392b', '#27ae60', '#7f8c8d', '#e67e22'

---
## Section 1: Kalla and Broockman (2018) meta-analysis [REAL]

**Data:** actual meta-analysis data from Kalla and Broockman's (2018, APSR) replication package (Harvard Dataverse `doi:10.7910/DVN/RAMHWP`). 51 experiments of candidate-campaign contact pulled from 21 original + literature; each row is one experimental estimate in percentage points on vote choice.

The meta-analytic question: what is the POOLED effect across these 51 experiments, and is it distinguishable from zero?

They use a **DerSimonian-Laird random-effects model**. The intuition:
- Each study has its own sampling variance (from its own N and design)
- There may be additional between-study heterogeneity (different campaigns, contact modes, voter populations)
- DL estimates both components and combines them into a pooled estimate

We load their actual data and run DL ourselves.

In [ ]:
# [REAL] Load Kalla-Broockman (2018) meta-analysis data from Dataverse
# Columns: study (citation), primary_or_general, seat, treatment_mode,
#          estimate_pp, se_pp, year
kb_url = 'https://raw.githubusercontent.com/Persuasion-at-Scale/effects-to-decisions/main/data/kalla-broockman-meta-analysis.csv'
raw = pd.read_csv(kb_url).dropna(subset=['estimate_pp', 'se_pp'])
# Drop SE=0 rows (would give infinite weight)
raw = raw[raw['se_pp'] > 0].reset_index(drop=True)

# Paper headline uses GENERAL elections; primaries behave differently
df = raw[raw['primary_or_general'] == 'General'].reset_index(drop=True).copy()
df['estimate'] = df['estimate_pp']
df['se'] = df['se_pp']
df['ci_lo'] = df['estimate'] - 1.96 * df['se']
df['ci_hi'] = df['estimate'] + 1.96 * df['se']

K = len(df)
print(f"[REAL] {len(raw)} total experiments in Kalla-Broockman meta-analysis")
print(f"       {K} are GENERAL election contests (the paper's headline subset)")
print(f"       {(raw['primary_or_general']=='Primary').sum()} Primary, excluded here")
print(f"Estimate range: [{df['estimate'].min():.3f}, {df['estimate'].max():.3f}] pp")
print(f"SE range:       [{df['se'].min():.3f}, {df['se'].max():.3f}] pp")
df.head()

In [ ]:
# DerSimonian-Laird random-effects meta-analysis
# Step 1: fixed-effects weights (inverse variance)
w_fe = 1.0 / df['se']**2

# Fixed-effects pooled estimate (used to compute Q)
theta_fe = np.average(df['estimate'], weights=w_fe)

# Step 2: compute Q statistic (test for heterogeneity)
Q = np.sum(w_fe * (df['estimate'] - theta_fe)**2)
k = len(df)
dof = k - 1

# Step 3: estimate between-study variance (tau-squared)
# DL estimator: tau^2 = max(0, (Q - df) / c) where c = sum(w) - sum(w^2)/sum(w)
c = w_fe.sum() - (w_fe**2).sum() / w_fe.sum()
tau_sq = max((Q - dof) / c, 0)

# Step 4: random-effects weights (inverse of within + between variance)
w_re = 1.0 / (df['se']**2 + tau_sq)

# Pooled random-effects estimate
theta_re = np.average(df['estimate'], weights=w_re)
se_re = np.sqrt(1.0 / w_re.sum())
ci_lo_re = theta_re - 1.96 * se_re
ci_hi_re = theta_re + 1.96 * se_re
p_val = 2 * (1 - stats.norm.cdf(abs(theta_re / se_re)))

# Heterogeneity statistics
I_sq = max(0, (Q - dof) / Q) * 100

print(f"=== DerSimonian-Laird Random-Effects Meta-Analysis ===")
print(f"Studies: k = {k}")
print(f"Pooled effect: {theta_re:.3f} pp (SE = {se_re:.3f})")
print(f"95% CI: [{ci_lo_re:.3f}, {ci_hi_re:.3f}] pp")
print(f"p-value: {p_val:.3f}")
print(f"")
print(f"Heterogeneity:")
print(f"  Q(df={dof}) = {Q:.2f}")
print(f"  tau^2 = {tau_sq:.3f}")
print(f"  I^2 = {I_sq:.1f}%")

In [ ]:
# Forest plot of the real 51 Kalla-Broockman experiments
fig, ax = plt.subplots(figsize=(10, 14))

# Sort by effect size for readability
df_sorted = df.sort_values('estimate').reset_index(drop=True)

for i, row in df_sorted.iterrows():
    ax.plot([row['ci_lo'], row['ci_hi']], [i, i],
            color=BLUE, linewidth=1.2, alpha=0.6)
    ax.plot(row['estimate'], i, 'o', color=BLUE, markersize=4)

# Meta-analytic diamond below all studies
diamond_y = len(df_sorted) + 2
ax.plot(theta_re, diamond_y, 'D', color='black', markersize=12, zorder=5)
ax.plot([ci_lo_re, ci_hi_re], [diamond_y, diamond_y], 'k-', linewidth=2.5)

# Zero line
ax.axvline(0, color=GRAY, linestyle='--', alpha=0.6)

ax.set_xlabel('Treatment Effect on Vote Choice (percentage points)', fontsize=12)
ax.set_title(
    f'[REAL] Kalla-Broockman 2018: {K} field experiments\n'
    f'Pooled effect = {theta_re:.2f} pp  '
    f'(95% CI [{ci_lo_re:.2f}, {ci_hi_re:.2f}], p = {p_val:.2f})',
    fontsize=13
)
ax.set_yticks([])

ax.annotate(
    'Pooled\n(DerSimonian-Laird)',
    xy=(theta_re, diamond_y),
    xytext=(max(df['estimate'].max(), 2) * 0.6, diamond_y - 8),
    fontsize=10,
    arrowprops=dict(arrowstyle='->', color='black')
)

plt.tight_layout()
plt.show()

**What you should see [REAL data]:** 44 individual effect estimates (blue dots) scattered near zero. Most 95% CIs cross zero. The meta-analytic diamond (black) sits essentially on zero with a very tight CI. Individual campaigns can show small positive or negative results due to noise, but the POOLED effect across all these real experiments is statistically indistinguishable from zero (p > 0.10 on General subset).

This is Kalla and Broockman's core finding, reproduced end-to-end from their replication data. The RCTs are not broken: they are correctly measuring a near-zero underlying effect. General-election persuasion from campaign contact is ~0.

**Where persuasion effects DO appear** (from the paper):
- Early primaries (before voters form opinions)
- Low-information races (school board, state legislature)
- Unusual candidates (partisan ID fails to predict)
- Early contact (weeks before election day)

---
## Section 2: Portable Power, Wheeler's formula on everyday A/B tests

Kalla-Broockman and Lewis-Rao both trace back to a single sample-size formula. Wheeler (1974) packaged $\alpha = 0.05$ and power $= 0.90$ into a portable rule:

$$N_{\text{per arm}} \;\approx\; \frac{16\,\sigma^2}{\Delta^2}$$

For a binary outcome (did they click / stop / wave), $\sigma^2 = \bar{p}(1-\bar{p})$ where $\bar{p}$ is the pooled response rate. Plug in the baseline and the absolute lift $\Delta$, get the per-arm N you need.

We run four A/B tests you already have intuition about:

1. **Ringtone pickup.** Pleasant ringtone vs jarring one. Do you answer?
2. **Wave back at a greeting.** Two different hellos on the street. Baseline wave-back rate $\approx 25\%$.
3. **Street survey stop rate.** Two openers. Baseline stop rate $\approx 1\%$.
4. **Ad click lift.** Two creatives. Baseline CTR $\approx 0.5\%$, realistic ~1% *relative* lift (Lewis-Rao setting).

Same formula. Required N spans five orders of magnitude.

In [ ]:
# Wheeler's portable power: N_per_arm = 16 * sigma^2 / delta^2
# (alpha = 0.05, power = 0.90, 2 arms)
def n_per_arm_bernoulli(p_A, p_B):
    """Wheeler's rule for Bernoulli outcomes, 2 arms, alpha=0.05, power=0.90."""
    p_bar = (p_A + p_B) / 2
    sigma2 = p_bar * (1 - p_bar)
    delta = abs(p_B - p_A)
    return 16 * sigma2 / delta**2

# Four A/B tests you have everyday intuition about
scenarios = [
    ('Ringtone pickup',       0.50,  0.75,    'Jarring ringtone vs pleasant one'),
    ('Wave back at hello',    0.25,  0.27,    'Two greetings, small lift on 25% base'),
    ('Street survey stop',    0.010, 0.012,   'Baseline 1% stop rate, 0.2pp lift'),
    ('Ad click (LR-style)',   0.005, 0.00505, 'Baseline 0.5% CTR, 1% relative lift'),
]

rows = []
for name, p_A, p_B, note in scenarios:
    n = n_per_arm_bernoulli(p_A, p_B)
    rows.append({
        'scenario': name,
        'p_A': p_A, 'p_B': p_B,
        'delta_pp': (p_B - p_A) * 100,
        'N_per_arm': int(round(n)),
        'N_total': int(round(2 * n)),
        'intuition': note,
    })
power_df = pd.DataFrame(rows)

print(power_df[['scenario','p_A','p_B','delta_pp','N_per_arm','N_total']].to_string(index=False))
print()
for r in rows:
    print(f"  {r['scenario']:<22} -> {r['intuition']}")

In [ ]:
# Bar chart: same formula, five orders of magnitude
fig, ax = plt.subplots(figsize=(10, 5))
colors_bars = [GREEN, BLUE, ORANGE, RED]
ax.barh(power_df['scenario'], power_df['N_per_arm'], color=colors_bars)
ax.set_xscale('log')
ax.set_xlabel('Sample size per arm (log scale)', fontsize=12)
ax.set_title(
    "Wheeler's formula applied to four A/B tests you have intuition about\n"
    "Same formula; N spans 5 orders of magnitude",
    fontsize=13
)
for i, n in enumerate(power_df['N_per_arm']):
    label = f'{n:,}' if n < 1_000_000 else f'{n/1e6:.1f}M'
    ax.text(n * 1.5, i, label, va='center', fontsize=11, fontweight='bold')
ax.grid(True, which='both', axis='x', alpha=0.3)
ax.set_xlim(10, 1e9)
plt.tight_layout()
plt.show()

**What you should see:** Sample-size requirements span from ~dozens (ringtone pickup) to tens of millions (ad click). Same formula, five orders of magnitude.

- **Ringtone & greeting:** cheap experiments. Any small team can run these to usable precision.
- **Street survey:** feasible but expensive. You need a big-city afternoon.
- **Ad click:** Lewis-Rao territory. The formula tells you *up front* that you need tens of millions of impressions per arm, so any small-scale "does this ad work?" experiment is a waste of everyone's time.

Next section watches this play out empirically.

### Interactive: your own power calculator

Wheeler's formula IS industry standard. Every major experimentation platform ships one:

- **Evan Miller** (free, the canonical reference): https://www.evanmiller.org/ab-testing/sample-size.html
- **Statsig** (acquired by OpenAI 2025): https://www.statsig.com/calculator
- **Optimizely, VWO, Eppo, Adobe Target**: all ship equivalent calculators inside their products

Below is a replication: give it a baseline rate, a minimum effect you want to detect, and your alpha/power targets, and it returns the per-arm N.

In [ ]:
# Classic two-proportion z-test sample size formula (generalizes Wheeler to any alpha/power)
# N_per_arm = ((z_alpha/2 + z_beta)^2 * (p_A*(1-p_A) + p_B*(1-p_B))) / (p_B - p_A)^2
def sample_size_calculator(baseline_pct, mde_pct_absolute, alpha=0.05, power=0.90):
    """
    baseline_pct:        current conversion rate, in percent (e.g. 5.0 for 5%)
    mde_pct_absolute:    minimum detectable absolute lift, in percentage points
                         (e.g. 1.0 for a lift from 5% to 6%)
    alpha:               significance level (default 0.05)
    power:               desired statistical power (default 0.90)

    Returns: (N_per_arm, N_total, z_alpha, z_beta)
    """
    from scipy.stats import norm
    p_A = baseline_pct / 100.0
    p_B = (baseline_pct + mde_pct_absolute) / 100.0
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    numerator = (z_alpha + z_beta) ** 2 * (p_A * (1 - p_A) + p_B * (1 - p_B))
    denominator = (p_B - p_A) ** 2
    n_per_arm = numerator / denominator
    return int(round(n_per_arm)), int(round(2 * n_per_arm)), z_alpha, z_beta

# Try it
print("Example 1: Baseline 5%, detect 1pp lift (5% -> 6%), alpha=0.05, power=0.90")
n, n_tot, za, zb = sample_size_calculator(5.0, 1.0)
print(f"  N per arm = {n:,}  |  Total = {n_tot:,}  |  z_alpha={za:.2f}  z_beta={zb:.2f}")

print("\nExample 2: Baseline 25% (greeting wave-back), detect 2pp lift")
n, n_tot, _, _ = sample_size_calculator(25.0, 2.0)
print(f"  N per arm = {n:,}  |  Total = {n_tot:,}")

print("\nExample 3: Baseline 0.5% (ad CTR), detect 0.005pp absolute lift")
n, n_tot, _, _ = sample_size_calculator(0.5, 0.005)
print(f"  N per arm = {n:,}  |  Total = {n_tot:,}")

print("\nExample 4: Your numbers here")
print("  Change the call below and re-run the cell:")
print("  sample_size_calculator(baseline_pct=10, mde_pct_absolute=2, alpha=0.05, power=0.80)")
n, n_tot, _, _ = sample_size_calculator(baseline_pct=10, mde_pct_absolute=2, alpha=0.05, power=0.80)
print(f"  -> N per arm = {n:,}  |  Total = {n_tot:,}")

---
## Section 3: Lewis-Rao (2015) power simulation [SIMULATED]

**Data:** synthetic consumer purchase outcomes, parameterized to Lewis-Rao's reported spikiness (most users spend $0, a small fraction spends a lot, lognormal tail). Not the actual paper data (proprietary, not released).

Kalla-Broockman measured effects that happen to be zero. Lewis-Rao show a different failure mode: even when effects might be nonzero, the NOISE is so large that we cannot measure them reliably.

Their setup: 25 digital ad experiments across major retailers. Total ad spend: $2.8M. Sample sizes: millions of person-weeks each. Outcome: consumer spending (which is extremely spiky: most people buy nothing, some buy a lot).

They show that the 95% confidence interval on ROI stays wider than 100 percentage points even with these huge samples. A CI of that width cannot distinguish "+50% profit" from "-50% profit." The experiment is statistically helpless.

Below we reproduce the key logic with synthetic data.

In [ ]:
# [SIMULATED] Per-user consumer outcomes with realistic spiky distribution
# Most users spend $0; a small fraction spends a lot (lognormal tail)
np.random.seed(3)  # Section 3 seed; makes this section standalone

def generate_users(n, lift_pct=0.0):
    """[SIMULATED] n users' purchase outcomes with optional treatment lift."""
    purchase_prob = 0.03  # ~3% buy anything
    effective_prob = purchase_prob * (1.0 + lift_pct / 100.0)
    buys = np.random.binomial(1, effective_prob, n)
    log_means = np.random.normal(np.log(30), 1.0, n)  # median ~$30, heavy tail
    purchases = np.exp(log_means) * buys
    return purchases

def run_experiment(n_per_arm, true_lift_pct=0.5):
    ctrl = generate_users(n_per_arm, lift_pct=0.0)
    treat = generate_users(n_per_arm, lift_pct=true_lift_pct)
    lift_hat = (treat.mean() - ctrl.mean()) / ctrl.mean() * 100.0
    se_diff = np.sqrt(treat.var()/n_per_arm + ctrl.var()/n_per_arm)
    se_lift = se_diff / ctrl.mean() * 100.0
    return lift_hat, se_lift

# Sanity check at 100K per arm
lift, se = run_experiment(100_000, true_lift_pct=0.5)
print(f"[SIMULATED] N = 100K per arm, true lift = 0.5%")
print(f"  Estimated lift: {lift:.2f}% (SE = {se:.2f}%)")
print(f"  95% CI: [{lift - 1.96*se:.2f}%, {lift + 1.96*se:.2f}%]")
print(f"  CI width: {2*1.96*se:.1f} percentage points")

In [ ]:
# Sweep sample sizes from 10K to 10M per arm
# At each N, run a single experiment and record the CI width
sample_sizes = [10_000, 30_000, 100_000, 300_000, 1_000_000, 3_000_000, 10_000_000]
results = []

for n in sample_sizes:
    lift_hat, se_lift = run_experiment(n, true_lift_pct=0.5)
    ci_width = 2 * 1.96 * se_lift
    results.append({'n_per_arm': n, 'lift_hat': lift_hat,
                    'se': se_lift, 'ci_width': ci_width})

result_df = pd.DataFrame(results)
print("Sample size sweep (true lift = 0.5%):")
print(result_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# Plot CI width vs sample size (log-log)
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(result_df['n_per_arm'], result_df['ci_width'],
          'o-', color=RED, markersize=10, linewidth=2, label='95% CI width (pp)')

# Reference: true lift of 0.5% (what we are trying to detect)
ax.axhline(0.5, color=GREEN, linestyle='--', linewidth=2,
           label='True lift = 0.5% (target to detect)')

# Reference: 100pp threshold (Lewis-Rao headline)
ax.axhline(100, color=GRAY, linestyle=':', linewidth=1.5,
           label='100 pp (Lewis-Rao headline)')

ax.set_xlabel('Sample size per arm', fontsize=12)
ax.set_ylabel('95% CI width (percentage points)', fontsize=12)
ax.set_title(
    '[SIMULATED] The Unfavorable Economics of Measuring Ad Returns\n'
    'CI width shrinks slowly; you need 10M+ per arm before it rivals the effect',
    fontsize=13
)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

**What you should see:** The 95% CI width drops as you add sample size, but it drops slowly (as $1/\sqrt{N}$). Even at 10M per arm, the CI width is still ~1 percentage point: still wider than the 0.5 pp effect we are trying to detect.

This is why Google's own ad team (Lewis and Rao both worked there) could not measure their own ad ROI reliably even with enormous samples.

Apply Fisher's N ≈ 16s²/d² rule of thumb: with d ≈ 0.001 (tiny standardized effect), you need roughly 16/0.000001 = 16 million per arm. Matches the empirical result above.

**Two failure modes from today:**

| Paper | Failure mode | Claim |
|---|---|---|
| Kalla-Broockman | empirical | we CAN measure; answer is ~0 |
| Lewis-Rao | epistemic | noise too big; we cannot measure |

---
## Section 4: Adaptive bandits, real data from Offer-Westort, Coppock & Green (2021) [REAL]

**Data:** actual experiment data from Offer-Westort, Coppock & Green's (2021, AJPS) Study 1, loaded from Harvard Dataverse (`doi:10.7910/DVN/CMUHBU`). 1000 MTurk participants, 10 batches of ~100, 8 right-to-work messaging arms. The authors ran a Thompson-sampling bandit: each batch's allocation depended on posteriors updated from prior batches' outcomes.

Both nulls above assume STATIC A/B: 50/50 allocation for the entire experiment. OW argue this wastes data. If one arm is clearly winning by batch 3, why keep sending 1/8 of traffic to each losing arm? Thompson sampling shifts traffic to arms that look good, stays on losers only long enough to rule them out.

We watch the real allocation evolve batch by batch. You should see Proposal 4 (BM) concentrating traffic as the algorithm learns it is the winner.

This is the bridge to Class 21 (Multi-Armed Bandits) and HW4.

In [ ]:
# [REAL] Load Offer-Westort Study 1 data from Harvard Dataverse
# 8-arm bandit on right-to-work messaging, 10 batches, 1000 MTurk participants
data_url = 'https://raw.githubusercontent.com/Persuasion-at-Scale/effects-to-decisions/main/data/offer-westort-2021-study1.csv'
ow = pd.read_csv(data_url)

print(f"[REAL] Loaded: {len(ow)} participants, {ow['batch'].nunique()} batches, "
      f"{ow['Z_rtw'].nunique()} RTW arms")
print(f"Columns: {list(ow.columns)}")
print()

# Arm-level success rates (binary agreement outcome)
arm_stats = ow.groupby('Z_rtw').agg(
    n=('Y_rtw', 'size'),
    success_rate=('Y_rtw', 'mean')
).sort_values('n', ascending=False)
print("Arm allocation + success rate (sorted by final allocation):")
print(arm_stats.round(3))

In [ ]:
# [REAL] Plot fraction of each batch assigned to each arm
alloc = ow.groupby(['batch', 'Z_rtw']).size().unstack(fill_value=0)
alloc_frac = alloc.div(alloc.sum(axis=1), axis=0)

# Identify winner for highlight
winner = arm_stats.index[0]

fig, ax = plt.subplots(figsize=(11, 6))
for arm in alloc_frac.columns:
    is_winner = (arm == winner)
    ax.plot(alloc_frac.index, alloc_frac[arm],
            marker='o', linewidth=(3 if is_winner else 1.2),
            alpha=(1.0 if is_winner else 0.45),
            color=(GREEN if is_winner else GRAY),
            label=arm if is_winner else None,
            zorder=(5 if is_winner else 2))

ax.axhline(1/8, color=RED, linestyle='--', linewidth=2,
           label='Static uniform (1/8 = 0.125)')
ax.set_xlabel('Batch (each ~100 participants)', fontsize=12)
ax.set_ylabel('Fraction of batch assigned to arm', fontsize=12)
ax.set_title(
    '[REAL] Offer-Westort Study 1: adaptive allocation across 10 batches\n'
    f'Thompson sampling concentrated traffic on {winner}',
    fontsize=13
)
ax.set_ylim(0, 1.0)
ax.legend(loc='center left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBy batch 10, Thompson sampling was sending "
      f"{alloc_frac.iloc[-1][winner]:.0%} of traffic to {winner}")
print(f"Static uniform would have sent 12.5% to each arm.")
print(f"Cumulative total: {arm_stats.loc[winner,'n']}/{len(ow)} = "
      f"{arm_stats.loc[winner,'n']/len(ow):.0%} of all 1000 participants.")

**What you should see [REAL data]:** In early batches (1-3), Thompson sampling allocates participants roughly uniformly across the 8 arms (~12-25% each) because posteriors are still flat. As evidence accumulates that Proposal 4 (BM) has a higher success rate, the algorithm concentrates subsequent batches on it, reaching 80-95% of traffic by the final batches. Static uniform allocation (red dashed line at 12.5%) is oblivious to the evidence and wastes participants on losing arms.

Same total budget of 1000 participants, many more hits on the winner. That is the adaptive move that Offer-Westort, Coppock, and Green advocate, demonstrated on their actual Study 1 data.

**Next week (Class 20 Targeting):** what if we do not just want the best AVERAGE arm, but the best arm FOR THIS PERSON? Answer: learn a model of $f_a(X) = E[Y \mid A=a, X]$ and pick $\arg\max_a f_a(X)$ per person.

**Class 21 (Multi-Armed Bandits):** formalize this. HW4 launches.

---

## Key takeaways

1. **Kalla-Broockman [REAL]:** 44 real general-election experiments give pooled persuasion effect ~0.005 pp (p > 0.10). The RCT works; the effect IS zero.
2. **Wheeler Portable Power [FORMULA]:** one formula predicts when A/B will be cheap (ringtone, greeting) and when it will be hopeless (ad click). Required N spans five orders of magnitude on the *same* formula.
3. **Lewis-Rao [SIMULATED]:** even with millions of person-weeks, ROI CIs can stay wider than the effects you want to measure. Power is unfavorable.
4. **Offer-Westort, Coppock & Green [REAL]:** static A/B wastes data. Real Study 1 bandit shifted 72% of 1000 participants to the winning arm by batch 10.

These ideas set up the next six classes: targeting, bandits, contextual bandits, recommender systems, LLMs, and governance.